In [1]:
import os

In [2]:
%pwd

'c:\\Users\\temmy\\OneDrive\\Desktop\\mlUdemy\\dscProject\\research'

In [5]:
os.chdir('../')
%pwd

'c:\\Users\\temmy\\OneDrive\\Desktop\\mlUdemy\\dscProject'

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str

In [7]:
from src.dscProject.constants import *
from src.dscProject.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(self,
                config_filepath=CONFIG_FILE_PATH,
                params_filepath=PARAMS_FILE_PATH,
                schema_filepath=SCHEMA_FILE_PATH
                ):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path,
            model_name=config.model_name,
            alpha=params.alpha,
            l1_ratio=params.l1_ratio,
            target_column=schema.name
        )

        return model_trainer_config

In [9]:
import pandas as pd
import os
from src.dscProject import logger
from sklearn.linear_model import ElasticNet
import joblib

In [ ]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train_model(self):
        logger.info("Loading training data")
        train_data = pd.read_csv(self.config.train_data_path)

        logger.info("Splitting features and target variable")
        X_train = train_data.drop(columns=[self.config.target_column], axis=1)
        y_train = train_data[self.config.target_column]

        logger.info("Training the ElasticNet model")
        model = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio, random_state=42)
        model.fit(X_train, y_train)

        model_dir = os.path.join(self.config.root_dir, "model")
        create_directories([model_dir])

        model_path = os.path.join(model_dir, self.config.model_name)
        joblib.dump(model, model_path)
        logger.info(f"Model saved at {model_path}")

In [12]:
try:
    config_manager = ConfigurationManager()
    model_trainer_config = config_manager.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train_model()
except Exception as e:
    logger.exception(e)
    raise e

[2026-06-02 22:03:39,711: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-02 22:03:39,719: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-02 22:03:39,730: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-06-02 22:03:39,734: INFO: common: created directory at: artifacts_root]
[2026-06-02 22:03:39,737: INFO: 2583058951: Loading training data]


[2026-06-02 22:03:39,957: INFO: 2583058951: Splitting features and target variable]
[2026-06-02 22:03:40,006: INFO: 2583058951: Training the ElasticNet model]
[2026-06-02 22:03:40,130: ERROR: 951467416: [Errno 2] No such file or directory: 'artifacts/model_trainer\\model\\model.joblib']
Traceback (most recent call last):
  File "C:\Users\temmy\AppData\Local\Temp\ipykernel_34344\951467416.py", line 5, in <module>
    model_trainer.train_model()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\temmy\AppData\Local\Temp\ipykernel_34344\2583058951.py", line 20, in train_model
    joblib.dump(model, model_path)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "c:\Users\temmy\OneDrive\Desktop\mlUdemy\dscProject\venv\Lib\site-packages\joblib\numpy_pickle.py", line 599, in dump
    with open(filename, "wb") as f:
         ~~~~^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'artifacts/model_trainer\\model\\model.joblib'


FileNotFoundError: [Errno 2] No such file or directory: 'artifacts/model_trainer\\model\\model.joblib'